# Ingestion Pipeline — Live Walkthrough

This notebook runs the **real** ingestion pipeline against a Jira ticket id. No mocks. Every cell executes against:

* Jira REST (`JIRA_BASE_URL`, `JIRA_TOKEN`)
* The IDP LLM gateway (`IDP_*` settings + `gpt-5-mini-idp`)
* The Azure embedding service (`text-embedding-3-large` via the embedding client)

It does **not** write to disk, FAISS, or Azure AI Search — the last cell only prints the upload payloads that *would* be sent.

Companion reference: [docs/ingestion.md](ingestion.md).

## Stages

1. Setup — imports, async glue, runtime config, clients
2. Fetch — `JiraTicketClient.get_ticket_data`
3. Label gate — refuse if no resolved value streams
4. Consolidate — `consolidate_ticket_text`
5. LLM #1 — structured summary (`summarize_ticket`)
6. LLM #2 — direct/implied classification (`classify_ticket_value_streams`)
7. Embed — `format_structured_summary_text` → `embed_batch`
8. Assemble — `TicketSummaryDocument.to_index_doc()`
9. Sink previews — FAISS metadata + Azure document shape
10. Teardown — close the Jira client

## 1. Setup

In [ ]:
# Prereqs:
#   * Run from the repo root so `src/` is importable.
#   * .env (or process env) must contain JIRA_BASE_URL, JIRA_TOKEN, and the
#     IDP / Azure credentials the project normally uses.
#   * `nest_asyncio` lets us `await` directly in notebook cells while still
#     reusing one async client across cells.

from __future__ import annotations

import asyncio
import base64
import json
import logging
import sys
from pathlib import Path
from pprint import pprint

import nest_asyncio
nest_asyncio.apply()

REPO = Path.cwd().resolve()
while REPO != REPO.parent and not (REPO / 'src' / 'vs_app').exists():
    REPO = REPO.parent
if str(REPO / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
logging.getLogger('httpx').setLevel(logging.WARNING)
logging.getLogger('httpcore').setLevel(logging.WARNING)

print('repo root:', REPO)

In [ ]:
# The one knob you change.
TICKET_ID = 'IDMT-19761'

In [ ]:
# Real runtime config (same one the batch wrapper uses for the primary profile).
from vs_app.container import build_ticket_fetcher
from vs_app.integrations.clients.embedding import EmbeddingClient
from vs_app.integrations.clients.llm import IDPChatOpenAI, build_extra_body
from vs_app.jobs.jira_batch.runtime.runtime_factory import build_ingestion_config
from vs_app.settings import EMBEDDING_DIMENSION, EMBEDDING_MODEL

LLM_MODEL = 'gpt-5-mini-idp'
REASONING_EFFORT = 'medium'

cfg = build_ingestion_config(
    llm_model=LLM_MODEL,
    embedding_model=EMBEDDING_MODEL,
    llm_max_output_tokens=None,
    summary_input_char_limit=20_000,
    classification_input_char_limit=20_000,
    skip_llm_summary=False,
    skip_llm_keywords=False,
    skip_llm_derived=False,
)

llm_client = IDPChatOpenAI(
    model=LLM_MODEL,
    extra_body=build_extra_body(reasoning_effort=REASONING_EFFORT),
)
embedding_client = EmbeddingClient(model=EMBEDDING_MODEL, dimension=EMBEDDING_DIMENSION)

print(f'llm_model         = {LLM_MODEL!r}, reasoning_effort = {REASONING_EFFORT!r}')
print(f'embedding_model   = {EMBEDDING_MODEL!r}, dimension = {EMBEDDING_DIMENSION}')
print(f'summary_input_chars       = {cfg.summary_input_char_limit}')
print(f'classification_input_chars= {cfg.classification_input_char_limit}')
print(f'max_documents             = {cfg.max_documents}, max_slides = {cfg.max_slides}, max_pages = {cfg.max_pages}')

In [ ]:
# Open the Jira async client once. We keep it open across cells using manual
# __aenter__ / __aexit__ so each stage is its own cell. The teardown cell at
# the bottom must run to release the underlying httpx.AsyncClient.

jira_ctx = build_ticket_fetcher(source='jira', verify_ssl=False)
jira_client = await jira_ctx.__aenter__()
print('jira_client opened:', type(jira_client).__name__)

## 2. Fetch — `JiraTicketClient.get_ticket_data`

Real call: hits Jira REST v2 for the issue, then runs `build_ticket_payload` (extracts attachments, runs `extract_themes(issuelinks)`, and resolves value-stream names/ids). Output is the canonical `ticket_data` dict every downstream stage consumes.

In [ ]:
ticket_data = await jira_client.get_ticket_data(TICKET_ID, config=cfg, llm_client=llm_client)

fields = ticket_data.get('fields') or {}
print(f"key          : {ticket_data.get('key')}")
print(f"summary      : {fields.get('summary')}")
print(f"status       : {(fields.get('status') or {}).get('name')}")
print(f"attachments  : {len(ticket_data.get('attachments') or [])}")
print(f"issuelinks   : {len(fields.get('issuelinks') or [])}")
print(f"themes       : {[t.get('key') for t in ticket_data.get('themes') or []]}")
print(f"value_stream_names : {ticket_data.get('value_stream_names')}")
print(f"value_stream_ids   : {ticket_data.get('value_stream_ids')}")
print(f"jira_group_ids     : {ticket_data.get('jira_group_ids')}")
print(f"label_source       : {ticket_data.get('value_stream_label_source')}")

## 3. Label gate

Calls the same `ensure_value_stream_labels` the batch job uses: if labels are missing it falls back to `resolve_value_stream_mapping` against `issuelinks`, then raises if still empty.

In [ ]:
import importlib.util

# `jobs/` is a top-level scripts dir, not a package — load it directly.
# We must register the module in sys.modules BEFORE exec_module, because
# `@dataclass(frozen=True)` inside the job file does sys.modules.get(cls.__module__)
# and would otherwise hit AttributeError: 'NoneType' object has no attribute '__dict__'.
spec = importlib.util.spec_from_file_location(
    'ingest_tickets_job', REPO / 'jobs' / 'ingest_tickets.py'
)
ingest_tickets_job = importlib.util.module_from_spec(spec)
sys.modules['ingest_tickets_job'] = ingest_tickets_job
spec.loader.exec_module(ingest_tickets_job)

ingest_tickets_job.ensure_value_stream_labels(ticket_data, llm_client=llm_client)

names = list(ticket_data.get('value_stream_names') or [])
ids = list(ticket_data.get('value_stream_ids') or [])
if not names and not ids:
    raise RuntimeError(
        f'{TICKET_ID} has no official Jira Theme value-stream labels; '
        f'refusing to index unlabeled ticket'
    )

print(f'ACCEPT: {len(names)} value-stream label(s)')
for n, vsid in zip(names, ids):
    print(f'  - {n}  ({vsid})')

## 4. Text consolidation

Real `consolidate_ticket_text`: assembles `[DESCRIPTION]` + ranked attachment extracts + top substantive `[COMMENT N]` blocks. Attachments are downloaded and parsed via the actual `pptx_extractor` / `pdf_extractor` / `docx_extractor` paths. Progress lines (ATTACHMENT attempting / accepted / skipped …) come from the consolidator itself.

In [ ]:
from vs_app.ingestion.summary.text_consolidator import consolidate_ticket_text

_progress_lines = []
def _progress(msg: str) -> None:
    _progress_lines.append(msg)
    print(f'  [{TICKET_ID}] {msg}')

consolidated_text = await consolidate_ticket_text(
    ticket_data,
    jira_client,
    cfg,
    progress=_progress,
)

print(f'\nconsolidated_text: {len(consolidated_text):,} chars')
print('\n--- first 1500 chars ---')
print(consolidated_text[:1500])
if len(consolidated_text) > 1500:
    print('\n--- last 600 chars ---')
    print(consolidated_text[-600:])

## 5. LLM #1 — structured summary

Real `summarize_ticket` call. The prompt is the one in [prompt_yaml/retrieval_summary.yaml](../prompt_yaml/retrieval_summary.yaml). On a content-filter error this auto-retries once with sanitized input.

In [ ]:
from vs_app.ingestion.summary.llm_summary_extractor import (
    classify_ticket_value_streams,
    summarize_ticket,
)

# `summarize_ticket` is sync but runs an LLM call internally. Off-load to a
# thread so a slow gateway doesn't freeze the notebook event loop.
summary_doc = await asyncio.to_thread(
    summarize_ticket,
    TICKET_ID,
    consolidated_text,
    llm_client,
    cfg,
)

print(f'ticket_id           : {summary_doc.ticket_id}')
print()
print(f'summary_text        :\n  {summary_doc.summary_text}')
print(f'\nbusiness_problem    :\n  {summary_doc.business_problem}')
print(f'\nbusiness_capability :\n  {summary_doc.business_capability}')
print(f'\nstakeholders        : {summary_doc.stakeholders}')
print(f'systems_and_products: {summary_doc.systems_and_products}')
print(f'key_terms           : {summary_doc.key_terms}')

## 6. LLM #2 — direct / implied classification

Real `classify_ticket_value_streams`. Forces the LLM to label each verified Jira value stream as **direct** or **implied** with a one-sentence reason. Strict completeness raises if any input label is unclassified.

In [ ]:
# Carry the verified labels onto the document, then classify.
summary_doc.value_stream_ids   = list(ticket_data.get('value_stream_ids') or [])
summary_doc.value_stream_names = list(ticket_data.get('value_stream_names') or [])
summary_doc.jira_group_ids     = list(ticket_data.get('jira_group_ids') or [])
summary_doc.label_source       = str(ticket_data.get('value_stream_label_source') or 'jira_issuelinks')

summary_doc.value_streams = await asyncio.to_thread(
    classify_ticket_value_streams,
    ticket_id=TICKET_ID,
    consolidated_text=consolidated_text,
    value_stream_ids=summary_doc.value_stream_ids,
    value_stream_names=summary_doc.value_stream_names,
    jira_group_ids=summary_doc.jira_group_ids,
    label_source=summary_doc.label_source,
    llm_client=llm_client,
    cfg=cfg,
)

summary_doc.direct_vs_names = [
    row['vs_name'] for row in summary_doc.value_streams
    if row.get('inference_type') == 'direct' and row.get('vs_name')
]
summary_doc.implied_vs_names = [
    row['vs_name'] for row in summary_doc.value_streams
    if row.get('inference_type') == 'implied' and row.get('vs_name')
]

print(f'direct_vs_names  : {summary_doc.direct_vs_names}')
print(f'implied_vs_names : {summary_doc.implied_vs_names}')
print()
for row in summary_doc.value_streams:
    print(f"  [{row['inference_type']:7}] {row['vs_name']}")
    print(f"            reason: {row['reason']}")

## 7. Embed

Format the structured summary the same way ingestion / FAISS / Azure all do, then embed with the real `EmbeddingClient` (model + dimension come from `vs_app.settings` — currently `text-embedding-3-small-idp`, 1536 dims).

In [ ]:
from vs_app.integrations.embeddings.client import embed_batch
from vs_app.ingestion.summary.mapper import format_structured_summary_text

embedding_text = format_structured_summary_text(summary_doc.to_index_doc())
print(f'embedding_text length = {len(embedding_text):,} chars\n')
print('--- embedding_text ---')
print(embedding_text)

vectors = await asyncio.to_thread(
    embed_batch,
    [embedding_text],
    embedding_client,
    EMBEDDING_MODEL,
)
summary_doc.summary_embedding = vectors[0]

vec = summary_doc.summary_embedding
norm = sum(v * v for v in vec) ** 0.5
print(f'\nvector dim = {len(vec)}, L2 norm = {norm:.4f}, first 6 floats = {[round(v, 4) for v in vec[:6]]}')

## 8. Assemble — `TicketSummaryDocument.to_index_doc()`

This is the exact dict that gets written into `summaries.json` and is the input to both FAISS and the Azure upload.

In [ ]:
index_doc = summary_doc.to_index_doc()

print('summaries.json record shape:')
for k, v in index_doc.items():
    if k == 'summary_embedding':
        print(f'  {k:22} <vector dim={len(v)}>')
    elif isinstance(v, list) and v and isinstance(v[0], dict):
        print(f'  {k:22} [{len(v)} rows]')
    elif isinstance(v, list):
        print(f'  {k:22} {v}')
    else:
        s = str(v)
        print(f'  {k:22} {s if len(s) <= 110 else s[:110] + "..."}')

## 9. Sink previews

Showing the **shapes** that would be written to each sink. No actual disk / network writes happen here — re-run via `py -3 jobs/ingest_tickets.py {TICKET_ID} --build-faiss --upload-azure` if you want persistence.

In [ ]:
# 9.1 FAISS — LangChain Document(page_content, metadata)
faiss_metadata = {
    'doc_type': 'summary',
    'ticket_id': index_doc['ticket_id'],
    'value_stream_names': index_doc['value_stream_names'],
    'value_stream_ids':   index_doc['value_stream_ids'],
    'direct_vs_names':    index_doc['direct_vs_names'],
    'implied_vs_names':   index_doc['implied_vs_names'],
    'label_source':       index_doc['label_source'],
}
print('FAISS Document.metadata:')
pprint(faiss_metadata)
print(f'\nFAISS Document.page_content length = {len(embedding_text):,} chars (identical to embedding_text above)')

In [ ]:
# 9.2 Azure AI Search — what build_historical_azure_documents emits per ticket
from vs_app.ingestion.persistence.azure_historical_index import build_historical_azure_documents

azure_docs, skipped = build_historical_azure_documents([index_doc], embedding=embedding_client)

print(f'docs to upload  : {len(azure_docs)}')
print(f'skipped         : {len(skipped)}')
if skipped:
    pprint(skipped)

if azure_docs:
    azure_doc = azure_docs[0]
    decoded = base64.urlsafe_b64decode(azure_doc['id'] + '=' * (-len(azure_doc['id']) % 4)).decode('utf-8')
    print(f"\nid (base64-urlsafe) : {azure_doc['id']}")
    print(f'decodes back to     : {decoded!r}')
    print()
    print('Top-level Azure document fields:')
    for k, v in azure_doc.items():
        if k == 'content_vector':
            print(f'  {k:22} <vector dim={len(v)}>')
        elif k == 'value_streams_json':
            print(f'  {k:22} (json string, {len(v):,} chars)')
        elif isinstance(v, list):
            print(f'  {k:22} {v}')
        else:
            s = str(v)
            print(f'  {k:22} {s if len(s) <= 110 else s[:110] + "..."}')

## 10. Teardown

Closes the underlying `httpx.AsyncClient`. **Always run this** before re-running the notebook from the top, otherwise you'll leak the connection.

In [ ]:
await jira_ctx.__aexit__(None, None, None)
print('jira_client closed.')

## What you just ran

| Stage | Real function called | Output |
| ----- | -------------------- | ------ |
| Fetch | `JiraTicketClient.get_ticket_data` | `ticket_data` dict |
| Label gate | `ensure_value_stream_labels` | accept / raise |
| Consolidate | `consolidate_ticket_text` | ≤20k-char blob |
| LLM #1 | `summarize_ticket` | `TicketSummaryDocument` (summary, problem, capability, facets) |
| LLM #2 | `classify_ticket_value_streams` | per-VS rows with `direct` / `implied` + reason |
| Embed | `format_structured_summary_text` + `embed_batch` | 3072-dim vector |
| Assemble | `TicketSummaryDocument.to_index_doc()` | `summaries.json` record |
| Azure shape | `build_historical_azure_documents` | upload-ready doc with base64 key |

To actually persist:

```bash
py -3 jobs/ingest_tickets.py IDMT-19761 --output-dir ticket_data --build-faiss
py -3 jobs/ingest_tickets.py IDMT-19761 --upload-azure --azure-document-action upload
```